In [14]:
from langgraph.graph import StateGraph,START,END
from langchain_ollama import ChatOllama,OllamaLLM
from typing import TypedDict 

In [15]:
model=ChatOllama(model="llama3.2")

In [16]:
class Blogstatereview:
    title:str
    outline:str
    blog:str
    evalute:int


In [17]:
def outline(state : Blogstatereview)->Blogstatereview:
    title=state['title']
    prompt=f"Generate a detailed outline for a log on the topic - {title}"
    outline=model.invoke(prompt)
    state['outline']=outline
    return state

In [18]:
def blog(state:Blogstatereview)->Blogstatereview:
    title=state['title']
    outline=state['outline']
    prompt=f'Write a detailed blog on the tittle {title} using the following outline \n {outline}'
    blog=model.invoke(prompt).content
    state['blog']=blog
    return state

In [24]:
def evalution(state):
    # 1. Grab whatever inputs you need from state
    blog_content = state.get('blog')
    
    # 2. Invoke your model to review the content
    # Assuming 'model' or 'llm' is your ChatOllama instance
    response = model.invoke(f"Review this blog: {blog_content}") 
    
    # 3. Save it back to state (YOUR FIX IS HERE)
    # Use string quotes for the key, and extract '.content' from the message
    state['evalution'] = response.content  
    
    return state



In [25]:
graph=StateGraph(Blogstatereview)
graph.add_node("outline",outline)
graph.add_node("blog",blog)
graph.add_node("evalution",evalution)
graph.add_edge(START,'outline')
graph.add_edge('outline','blog')
graph.add_edge('blog','evalution')
graph.add_edge('evalution',END)

workflow=graph.compile()

In [26]:
input_state={'title':"Rise of AI in India"}
output_state=workflow.invoke(input_state)
print(output_state)

{'title': 'Rise of AI in India', 'outline': AIMessage(content='Here is a detailed outline for a blog post on "The Rise of AI in India":\n\n**I. Introduction**\n\n* Brief overview of the importance of Artificial Intelligence (AI) in today\'s digital landscape\n* Explanation of the growing interest in AI in India and its potential impact on the country\'s economy, jobs, and society\n* Thesis statement: The rise of AI in India is transforming industries, revolutionizing businesses, and opening up new opportunities for growth and innovation.\n\n**II. Historical Context**\n\n* Overview of India\'s early experiments with AI (e.g., 1960s-1980s)\n* Discussion of the country\'s technological advancements during the 1990s-2000s (e.g., outsourcing, e-commerce)\n* Explanation of how India\'s IT sector became a hub for AI development in recent years\n\n**III. Current State of AI in India**\n\n* Overview of the current state of AI adoption in various industries (e.g., finance, healthcare, education)

In [27]:
import pprint

input_state = {'title': "Rise of AI in India"}
output_state = workflow.invoke(input_state)

print("🎯 FINAL GRAPH STATE:")
print("=" * 40)
pprint.pprint(output_state)


🎯 FINAL GRAPH STATE:
{'blog': "The Rise of AI in India: A New Era for the Country's Economic and "
         'Social Landscape\n'
         '\n'
         'Introduction\n'
         '\n'
         'Artificial Intelligence (AI) has become an integral part of our '
         'daily lives, transforming the way we live, work, and interact with '
         'each other. In recent years, AI has experienced a rapid growth in '
         "India, with far-reaching implications for the country's economic, "
         'social, and cultural landscape. This blog post aims to explore the '
         'rise of AI in India, its significance, and the opportunities and '
         'challenges it presents.\n'
         '\n'
         'Background and Context\n'
         '\n'
         "India's IT industry has been growing steadily over the years, with "
         'the country emerging as a major player in the global technology '
         'sector. The Indian government has also taken significant steps to '
         'promot